# YOLOv8 Traffic Violation Detection - Training on Kaggle
Train YOLOv8 model with traffic violation dataset (car, motorcycle, bicycle, truck, bus)

In [ ]:
# Cài đặt thư viện
!pip install ultralytics -q

In [ ]:
import os
import shutil
import random
from pathlib import Path

# Đường dẫn dataset trên Kaggle
BASE_PATH = '/kaggle/input/datasets/khoilam4444/traffic-violation-data/DATA'

TRAIN_IMG = os.path.join(BASE_PATH, 'TRAIN', 'Image')
TRAIN_LBL = os.path.join(BASE_PATH, 'TRAIN', 'Label')

print('Train images:', len(os.listdir(TRAIN_IMG)))
print('Train labels:', len(os.listdir(TRAIN_LBL)))
print('VALIDATION contents:', os.listdir(os.path.join(BASE_PATH, 'VALIDATION')))

In [ ]:
# Tạo cấu trúc thư mục chuẩn YOLO trong /kaggle/working/
WORK_DIR = '/kaggle/working/dataset'

for split in ['train', 'val']:
    os.makedirs(f'{WORK_DIR}/{split}/images', exist_ok=True)
    os.makedirs(f'{WORK_DIR}/{split}/labels', exist_ok=True)

# Lấy danh sách ảnh và shuffle, chia 80/20
all_imgs = sorted(Path(TRAIN_IMG).glob('*'))
random.seed(42)
random.shuffle(all_imgs)

split_idx = int(len(all_imgs) * 0.8)
train_imgs = all_imgs[:split_idx]
val_imgs   = all_imgs[split_idx:]

def copy_pair(img_path, dest_split):
    # Copy ảnh
    shutil.copy(img_path, f'{WORK_DIR}/{dest_split}/images/{img_path.name}')
    # Copy label tương ứng (đổi extension sang .txt)
    lbl_path = Path(TRAIN_LBL) / (img_path.stem + '.txt')
    if lbl_path.exists():
        shutil.copy(lbl_path, f'{WORK_DIR}/{dest_split}/labels/{lbl_path.name}')

for f in train_imgs:
    copy_pair(f, 'train')

for f in val_imgs:
    copy_pair(f, 'val')

print('Train images:', len(os.listdir(f'{WORK_DIR}/train/images')))
print('Val   images:', len(os.listdir(f'{WORK_DIR}/val/images')))

In [ ]:
# Tạo file cấu hình YAML
yaml_content = f"""path: {WORK_DIR}
train: train/images
val:   val/images

nc: 5
names:
  0: car
  1: motorcycle
  2: bicycle
  3: truck
  4: bus
"""

yaml_path = '/kaggle/working/traffic.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print('YAML saved to:', yaml_path)
print(yaml_content)

In [ ]:
from ultralytics import YOLO

# Load model YOLOv8s (small - phù hợp với Kaggle GPU)
model = YOLO('yolov8s.pt')

# Train
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # GPU
    workers=2,
    project='/kaggle/working/runs',
    name='traffic_detect',
    patience=10,        # Early stopping
    save=True,
    exist_ok=True
)

In [ ]:
# Đánh giá model trên validation set
metrics = model.val()
print(f'mAP50   : {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')

In [ ]:
# Kiểm tra đường dẫn best model
best_pt = '/kaggle/working/runs/traffic_detect/weights/best.pt'
print('Best model exists:', os.path.exists(best_pt))

# Copy best.pt ra thư mục gốc để dễ download
shutil.copy(best_pt, '/kaggle/working/best.pt')
print('Saved to /kaggle/working/best.pt')

In [ ]:
# Hiển thị biểu đồ training results
from IPython.display import Image, display
import glob

results_png = '/kaggle/working/runs/traffic_detect/results.png'
if os.path.exists(results_png):
    display(Image(results_png))